In [1]:
from tensorflow.keras.datasets import mnist
from variational_ae import VariationalAutoencoder
import numpy as np
from sklearn.model_selection import train_test_split

2025-08-03 17:43:03.541706: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-08-03 17:43:03.554262: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1754214183.568879   32166 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1754214183.573074   32166 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1754214183.583054   32166 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

In [2]:
(x_train_full, _), (x_test, _) = mnist.load_data()
x_all = np.concatenate([x_train_full, x_test], axis=0).astype("float32") / 255.0
x_all = x_all[..., np.newaxis]

x_train, x_val = train_test_split(x_all, test_size=0.05, random_state=42)

print("Train shape:", x_train.shape)
print("Validation shape:", x_val.shape)

Train shape: (66500, 28, 28, 1)
Validation shape: (3500, 28, 28, 1)


In [3]:
LEARNING_RATE = 0.0005
BATCH_SIZE = 32
EPOCHS = 50

In [4]:
input_shape = x_train.shape[1:]
latent_space_dim = 2
decoder_out_filter = 1

In [5]:
autoencoder = VariationalAutoencoder(input_shape, latent_space_dim, decoder_out_filter, conv_layers_config=[
    {'filters': 32, 'kernel_size': (3, 3), 'strides': (1, 1)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (2, 2)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (2, 2)},
    {'filters': 64, 'kernel_size': (3, 3), 'strides': (1, 1)}
])

I0000 00:00:1754214186.411517   32166 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 9711 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3060, pci bus id: 0000:01:00.0, compute capability: 8.6


In [6]:
autoencoder.compile(learning_rate=LEARNING_RATE)

In [7]:
# autoencoder.summary()

In [8]:
autoencoder.fit(
    x=x_train,
    y=x_train, # Autoencoders typically use the same data for input and output
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(x_val, x_val), # Validation data for monitoring
    shuffle=True
)

Epoch 1/50


OperatorNotAllowedInGraphError: Exception encountered when calling VariationalAutoencoder.call().

[1mIterating over a symbolic `tf.Tensor` is not allowed. You can attempt the following resolutions to the problem: If you are running in Graph mode, use Eager execution mode or decorate this function with @tf.function. If you are using AutoGraph, you can try decorating this function with @tf.function. If that does not work, then you may be using an unsupported feature or your source code may not be visible to AutoGraph. See https://github.com/tensorflow/tensorflow/blob/master/tensorflow/python/autograph/g3doc/reference/limitations.md#access-to-source-code for more information.[0m

Arguments received by VariationalAutoencoder.call():
  • inputs=tf.Tensor(shape=(None, 28, 28, 1), dtype=float32)

In [ ]:
autoencoder.save_all()  # Save the trained model